In [268]:
import numpy as np
import pandas as pd

In [269]:
data = pd.read_csv(
    "/home/julia/Рабочий стол/code/NSU_AI-Robotics/tvims/House-Prices.csv"
)
df = pd.DataFrame(data)
df["waterbody"] = df["waterbody"].fillna("None")
df =df.dropna()
df.head()

,price,resid_area,air_qual,room_num,age,dist1,dist2,dist3,dist4,teachers,poor_prop,airport,n_hos_beds,n_hot_rooms,waterbody,rainfall,bus_ter,parks,Sold
0,24.0,32.31,0.538,6.575,65.2,4.35,3.81,4.18,4.01,24.7,4.98,YES,5.480,11.1920,River,23,YES,0.049347,0
1,21.6,37.07,0.469,6.421,78.9,4.99,4.70,5.12,5.06,22.2,9.14,NO,7.332,12.1728,Lake,42,YES,0.046146,1
2,34.7,37.07,0.469,7.185,61.1,5.03,4.86,5.01,4.97,22.2,4.03,NO,7.394,101.1200,None,38,YES,0.045764,0
3,33.4,32.18,0.458,6.998,45.8,6.21,5.93,6.16,5.96,21.3,2.94,YES,9.268,11.2672,Lake,45,YES,0.047151,0
4,36.2,32.18,0.458,7.147,54.2,6.16,5.86,6.37,5.86,21.3,5.33,NO,8.824,11.2896,Lake,55,YES,0.039474,0


In [270]:
X = df.drop(columns=["price"])
y = df["price"]

# one hot encoding
X = pd.get_dummies(X, columns=["waterbody"], dtype=int)

X["bus_ter"] = (
    X["bus_ter"].replace({"YES": 1, "NO": 0})
)
X["airport"] = X["airport"].replace({"YES": 1, "NO": 0})
X.head()

/tmp/ipykernel_8252/1116865364.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X["bus_ter"].replace({"YES": 1, "NO": 0})
/tmp/ipykernel_8252/1116865364.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X["airport"] = X["airport"].replace({"YES": 1, "NO": 0})


,resid_area,air_qual,room_num,age,dist1,dist2,dist3,dist4,teachers,poor_prop,...,n_hos_beds,n_hot_rooms,rainfall,bus_ter,parks,Sold,waterbody_Lake,waterbody_Lake and River,waterbody_None,waterbody_River
0,32.31,0.538,6.575,65.2,4.35,3.81,4.18,4.01,24.7,4.98,...,5.480,11.1920,23,1,0.049347,0,0,0,0,1
1,37.07,0.469,6.421,78.9,4.99,4.70,5.12,5.06,22.2,9.14,...,7.332,12.1728,42,1,0.046146,1,1,0,0,0
2,37.07,0.469,7.185,61.1,5.03,4.86,5.01,4.97,22.2,4.03,...,7.394,101.1200,38,1,0.045764,0,0,0,1,0
3,32.18,0.458,6.998,45.8,6.21,5.93,6.16,5.96,21.3,2.94,...,9.268,11.2672,45,1,0.047151,0,1,0,0,0
4,32.18,0.458,7.147,54.2,6.16,5.86,6.37,5.86,21.3,5.33,...,8.824,11.2896,55,1,0.039474,0,1,0,0,0


In [271]:
X.shape[1]
X.shape[0]

498

1. Исключите сильно коррелированные независимые переменные(Х). Под «слишком высокой» корреляцией можно понимать абсолютное значение коэффициента корреляции выше 0.9. Это поможет избежать проблемы мультиколлинеарности в модели.

In [272]:
# корелляция из лекций
import numpy as np

def compute_correlation(x, y):
    n = len(x)
    mean_x = np.mean(x)
    mean_y = np.mean(y)
    numerator = np.sum((x - mean_x) * (y - mean_y))
    denominator = np.sqrt(np.sum((x - mean_x) ** 2) * np.sum((y - mean_y) ** 2))
    if denominator == 0:
        return 0.0  
    return numerator / denominator


n_features = X.shape[1]
to_drop = set()

for i in range(n_features):
    for j in range(i + 1, n_features): 
        corr = abs(compute_correlation(X.iloc[:, i].values, X.iloc[:, j].values))
        if corr > 0.9:
            to_drop.add(j)
X = X.drop(X.columns[list(to_drop)], axis=1)
# print(to_drop) - dist2	dist3	dist4   parks
X.head()

,resid_area,air_qual,room_num,age,dist1,teachers,poor_prop,airport,n_hos_beds,n_hot_rooms,rainfall,bus_ter,Sold,waterbody_Lake,waterbody_Lake and River,waterbody_None,waterbody_River
0,32.31,0.538,6.575,65.2,4.35,24.7,4.98,1,5.480,11.1920,23,1,0,0,0,0,1
1,37.07,0.469,6.421,78.9,4.99,22.2,9.14,0,7.332,12.1728,42,1,1,1,0,0,0
2,37.07,0.469,7.185,61.1,5.03,22.2,4.03,0,7.394,101.1200,38,1,0,0,0,1,0
3,32.18,0.458,6.998,45.8,6.21,21.3,2.94,1,9.268,11.2672,45,1,0,1,0,0,0
4,32.18,0.458,7.147,54.2,6.16,21.3,5.33,0,8.824,11.2896,55,1,0,1,0,0,0


In [273]:
X.shape[1]

17

3. Проверьте, не улучшится ли связь некоторых признаков с зависимой
переменной при применении различных преобразований (например,
логарифмирования или взятия квадратного корня). Если после пре-
образования переменной её корреляция с целевой переменной увели-
чивается, используйте преобразованную версию этой переменной.

In [274]:
X = X.copy()

transformations = [
    ("original", lambda x: x, lambda x: True),
    ("log",np.log,lambda x: np.all(x > 0),),  
    ("sqrt", np.sqrt, lambda x: np.all(x >= 0)),
    ("square", lambda x: x**2, lambda x: True),
]

categorical_dummies = [
    "airport",
    "bus_ter",
    "waterbody_Lake",
    "waterbody_River",
    "waterbody_Lake and River",
    "waterbody_None",
]

numeric_cols = [col for col in X.columns if col not in categorical_dummies]

for col in numeric_cols:  # те которые были за энкожены не трогаем
    x_original = X[col].values
    y_vals = y.values

    best_corr = abs(compute_correlation(x_original, y_vals))
    best_transformed = x_original
    best_name = "original"

    for name, func, condition in transformations:
        if not condition(x_original):
            continue
        try:
            x_trans = func(x_original)
            if np.any(~np.isfinite(x_trans)):
                continue
            corr = abs(compute_correlation(x_trans, y_vals))
            if corr > best_corr:
                best_corr = corr
                best_transformed = x_trans
                best_name = name
        except Exception:
            continue 

    if best_name != "original":
        print(
            f"Признак '{col}': лучшее преобразование = {best_name}, корреляция = {best_corr:.4f}"
        )
        X[col] = best_transformed
    else:
        print(f"Признак '{col}' лучше в оригинале")

X.head()

Признак 'resid_area': лучшее преобразование = log, корреляция = 0.4939
Признак 'air_qual': лучшее преобразование = log, корреляция = 0.4341
Признак 'room_num': лучшее преобразование = square, корреляция = 0.7221
Признак 'age': лучшее преобразование = square, корреляция = 0.3924
Признак 'dist1': лучшее преобразование = log, корреляция = 0.2889
Признак 'teachers': лучшее преобразование = log, корреляция = 0.5057
Признак 'poor_prop': лучшее преобразование = log, корреляция = 0.8183
Признак 'n_hos_beds': лучшее преобразование = square, корреляция = 0.1111
Признак 'n_hot_rooms': лучшее преобразование = square, корреляция = 0.0270
Признак 'rainfall': лучшее преобразование = log, корреляция = 0.0580
Признак 'Sold' лучше в оригинале


,resid_area,air_qual,room_num,age,dist1,teachers,poor_prop,airport,n_hos_beds,n_hot_rooms,rainfall,bus_ter,Sold,waterbody_Lake,waterbody_Lake and River,waterbody_None,waterbody_River
0,3.475377,-0.619897,43.230625,4251.04,1.470176,3.206803,1.605430,1,30.030400,125.260864,3.135494,1,0,0,0,0,1
1,3.612808,-0.757153,41.229241,6225.21,1.607436,3.100092,2.212660,0,53.758224,148.177060,3.737670,1,1,1,0,0,0
2,3.612808,-0.757153,51.624225,3733.21,1.615420,3.100092,1.393766,0,54.671236,10225.254400,3.637586,1,0,0,0,1,0
3,3.471345,-0.780886,48.972004,2097.64,1.826161,3.058707,1.078410,1,85.895824,126.949796,3.806662,1,0,1,0,0,0
4,3.471345,-0.780886,51.079609,2937.64,1.818077,3.058707,1.673351,0,77.862976,127.455068,4.007333,1,0,1,0,0,0


In [275]:
"это набросок"

"""
# потом если смотреть на формулы то у Х первый столбец единицы. это сделано для бета0
X_final = np.column_stack([np.ones(X_final.shape[0]), X_final.values])
X = X_final.values
# тогда надо найти сначала бета с крышечкой
beta_hat = np.linalg.solve(X.T @ X, X.T @ y)
# тут решается скорее (X.T@X)beta_hat = X.T@y
# но это то же самое что найти по изначальной формуле

# из нее найдем у с крышечкой
y_hat = X @ beta_hat
#и ошибки
eps_hat = y - y_hat
"""

'\n# потом если смотреть на формулы то у Х первый столбец единицы. это сделано для бета0\nX_final = np.column_stack([np.ones(X_final.shape[0]), X_final.values])\nX = X_final.values\n# тогда надо найти сначала бета с крышечкой\nbeta_hat = np.linalg.solve(X.T @ X, X.T @ y)\n# тут решается скорее (X.T@X)beta_hat = X.T@y\n# но это то же самое что найти по изначальной формуле\n\n# из нее найдем у с крышечкой\ny_hat = X @ beta_hat\n#и ошибки\neps_hat = y - y_hat\n'

In [ ]:
X = X
def lin_regression(feature_names, X, y):
    X_subset = X[feature_names].values
    y_vals = y.values

    n = X_subset.shape[0]
    k = len(feature_names)
    p = k + 1  # +1 для beta0

    X = np.column_stack([np.ones(n), X_subset])  

    beta_hat = np.linalg.solve(X.T @ X, X.T @ y_vals)

    y_hat = X @ beta_hat
    eps_hat = y_vals - y_hat

    # оценка дисперсии ошибок

    sigma2_hat = np.sum(eps_hat**2) / (n)

    return beta_hat, sigma2_hat, X

In [ ]:
def beta_hat_s_standartized(feature_names, s, X, y): 
    # feature_names - список признаков
    # s - номер признака по коорому ищем
    beta_hat, sigma2_hat, X = lin_regression(feature_names, X, y)
    XTX_inv = np.linalg.inv(X.T @ X)

    disp_beta_s = sigma2_hat * XTX_inv[s+1,s+1]
    
    # тк считаем что распределены нормально с мат ожиданием 0
    stand_beta_s = beta_hat[s + 1] / np.sqrt(disp_beta_s)
    return float(stand_beta_s)

In [278]:
from scipy.stats import norm

def find_p_value(feature_names, s, X, y):
    extr_mark = beta_hat_s_standartized(feature_names, s, X, y)
    # beta_hat_s_standartized распределено примерно как норм станд
    p_val = 2 * (1 - norm.cdf(abs(extr_mark)))
    # модуль значение
    # norm.cdf вычисляет вероятность что случайная величина норм станд распр < abs(extr_mark)
    # 1 - ... что мы попадем в хвост
    # тогда чтобы получить p-value нужно еще на 2 умножить
    return float(p_val)

1. Сначала обучите модель используя все доступные переменные;

In [279]:
X.head()

,resid_area,air_qual,room_num,age,dist1,teachers,poor_prop,airport,n_hos_beds,n_hot_rooms,rainfall,bus_ter,Sold,waterbody_Lake,waterbody_Lake and River,waterbody_None,waterbody_River
0,3.475377,-0.619897,43.230625,4251.04,1.470176,3.206803,1.605430,1,30.030400,125.260864,3.135494,1,0,0,0,0,1
1,3.612808,-0.757153,41.229241,6225.21,1.607436,3.100092,2.212660,0,53.758224,148.177060,3.737670,1,1,1,0,0,0
2,3.612808,-0.757153,51.624225,3733.21,1.615420,3.100092,1.393766,0,54.671236,10225.254400,3.637586,1,0,0,0,1,0
3,3.471345,-0.780886,48.972004,2097.64,1.826161,3.058707,1.078410,1,85.895824,126.949796,3.806662,1,0,1,0,0,0
4,3.471345,-0.780886,51.079609,2937.64,1.818077,3.058707,1.673351,0,77.862976,127.455068,4.007333,1,0,1,0,0,0


In [280]:
# короче если так не сделать оно улетит на выборе признаков потому что категориальные переенные или чего
# ничего не поняв но в интернете говорят так надо

X = X.drop(columns=["waterbody_Lake and River"])
all_features = X.columns.tolist()

beta_all, sigma2_all, _ = lin_regression(all_features, X, y)
print("Коэффициенты модели (включая 0):")
for i, name in enumerate(["intercept"] + all_features):
    print(f"{name:20}: {beta_all[i]:.4f}")

print(f"\nОценка дисперсии ошибок : {sigma2_all:.4f}")

Коэффициенты модели (включая 0):
intercept           : -10.6517
resid_area          : -2.1364
air_qual            : -11.1912
room_num            : 0.2481
age                 : 0.0001
dist1               : -6.2457
teachers            : 17.5446
poor_prop           : -9.1735
airport             : 0.9328
n_hos_beds          : 0.0198
n_hot_rooms         : -0.0001
rainfall            : -0.1449
bus_ter             : -0.0524
Sold                : -3.1815
waterbody_Lake      : -0.1361
waterbody_None      : 0.2082
waterbody_River     : -0.1922

Оценка дисперсии ошибок : 15.3714


In [281]:
X.head()

,resid_area,air_qual,room_num,age,dist1,teachers,poor_prop,airport,n_hos_beds,n_hot_rooms,rainfall,bus_ter,Sold,waterbody_Lake,waterbody_None,waterbody_River
0,3.475377,-0.619897,43.230625,4251.04,1.470176,3.206803,1.605430,1,30.030400,125.260864,3.135494,1,0,0,0,1
1,3.612808,-0.757153,41.229241,6225.21,1.607436,3.100092,2.212660,0,53.758224,148.177060,3.737670,1,1,1,0,0
2,3.612808,-0.757153,51.624225,3733.21,1.615420,3.100092,1.393766,0,54.671236,10225.254400,3.637586,1,0,0,1,0
3,3.471345,-0.780886,48.972004,2097.64,1.826161,3.058707,1.078410,1,85.895824,126.949796,3.806662,1,0,1,0,0
4,3.471345,-0.780886,51.079609,2937.64,1.818077,3.058707,1.673351,0,77.862976,127.455068,4.007333,1,0,1,0,0


In [282]:
cat_groups = {
    "waterbody": [
        "waterbody_Lake",
        "waterbody_River",
        "waterbody_Lake and River",
        "waterbody_None",
    ],
    "bus_ter": ["bus_ter"],
    "airport": ["airport"],
}
categorical_dummies = [col for cols in cat_groups.values() for col in cols]

current_features = X.columns.tolist()
print(len(current_features), current_features)

# повторять пока не все p-value < 0.05
iteration = 0
while True:
    iteration += 1
    print(f"\n=== Итерация {iteration} ===")
    print(f"Текущие признаки: {current_features}")

    # p-value для всех коэффициентов
    p_values = []
    for s in range(len(current_features)):
        p = find_p_value(current_features, s, X, y) 
        p_values.append(p)

    if all(p < 0.05 for p in p_values): 
        print(" коэффициенты значимы (p < 0.05). Остановка.")
        break

    # p-value по категориям
    feature_to_p = dict(zip(current_features, p_values))
    group_pvals = {}

    for base_name, cols in cat_groups.items():
        present_cols = [col for col in cols if col in current_features]
        if present_cols:
            min_p = min(feature_to_p[col] for col in present_cols)
            group_pvals[base_name] = min_p

    numeric_features = [
        col for col in current_features if col not in categorical_dummies
    ]
    for col in numeric_features:
        group_pvals[col] = feature_to_p[col]

    # саааамый плохой признак или группа
    worst_name, worst_p = max(group_pvals.items(), key=lambda item: item[1])
    print(f"Максимальное p-value = {worst_p:.4f} удаляем: {worst_name}")

    #если категориальная то удалить всю категорию 
    if worst_name in cat_groups:
        cols_to_remove = cat_groups[worst_name]
        for col in cols_to_remove:
            if col in current_features:
                current_features.remove(col)
        print(f" Удалены столбцы: {cols_to_remove}")
    else:
        # а если числовой то тлько его
        current_features.remove(worst_name)
        print(f"  Удалён признак: {worst_name}")

    if len(current_features) == 0:
        print("Все признаки удалены!")
        break

16 ['resid_area', 'air_qual', 'room_num', 'age', 'dist1', 'teachers', 'poor_prop', 'airport', 'n_hos_beds', 'n_hot_rooms', 'rainfall', 'bus_ter', 'Sold', 'waterbody_Lake', 'waterbody_None', 'waterbody_River']

=== Итерация 1 ===
Текущие признаки: ['resid_area', 'air_qual', 'room_num', 'age', 'dist1', 'teachers', 'poor_prop', 'airport', 'n_hos_beds', 'n_hot_rooms', 'rainfall', 'bus_ter', 'Sold', 'waterbody_Lake', 'waterbody_None', 'waterbody_River']
Максимальное p-value = 1.0000 удаляем: bus_ter
 Удалены столбцы: ['bus_ter']

=== Итерация 2 ===
Текущие признаки: ['resid_area', 'air_qual', 'room_num', 'age', 'dist1', 'teachers', 'poor_prop', 'airport', 'n_hos_beds', 'n_hot_rooms', 'rainfall', 'Sold', 'waterbody_Lake', 'waterbody_None', 'waterbody_River']


Максимальное p-value = 0.7735 удаляем: rainfall
  Удалён признак: rainfall

=== Итерация 3 ===
Текущие признаки: ['resid_area', 'air_qual', 'room_num', 'age', 'dist1', 'teachers', 'poor_prop', 'airport', 'n_hos_beds', 'n_hot_rooms', 'Sold', 'waterbody_Lake', 'waterbody_None', 'waterbody_River']
Максимальное p-value = 0.6903 удаляем: waterbody
 Удалены столбцы: ['waterbody_Lake', 'waterbody_River', 'waterbody_Lake and River', 'waterbody_None']

=== Итерация 4 ===
Текущие признаки: ['resid_area', 'air_qual', 'room_num', 'age', 'dist1', 'teachers', 'poor_prop', 'airport', 'n_hos_beds', 'n_hot_rooms', 'Sold']
Максимальное p-value = 0.7080 удаляем: n_hot_rooms
  Удалён признак: n_hot_rooms

=== Итерация 5 ===
Текущие признаки: ['resid_area', 'air_qual', 'room_num', 'age', 'dist1', 'teachers', 'poor_prop', 'airport', 'n_hos_beds', 'Sold']
Максимальное p-value = 0.4666 удаляем: age
  Удалён признак: age

=== Итерация 6 ===
Текущие признаки: ['resid_area', 'air_qual', 'room_num', 'dist1', 'tea